<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/STEP5_NUOVO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

In [4]:
!python3 -m pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 29.9 MB/s

In [4]:
!pip uninstall -y wandb
!pip install -U wandb

import wandb
from lightning.pytorch.loggers import WandbLogger

wandb.login()

wandb_logger = WandbLogger(project="cityscapes_finetuning",
                           name="finetune_selective_run",
                           log_model = False
                           )


Found existing installation: wandb 0.27.1
Uninstalling wandb-0.27.1:
  Successfully uninstalled wandb-0.27.1
  Using cached wandb-0.27.1-py3-none-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached wandb-0.27.1-py3-none-manylinux_2_28_x86_64.whl (26.4 MB)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: giuliadesantis (giuliadesantis-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Setup

In [5]:
import yaml
import torch
import importlib
import logging
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import warnings

from lightning import seed_everything
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from lightning.pytorch.callbacks import ModelCheckpoint

seed_everything(0, verbose=False)

device = 0
IGNORE_INDEX = 255
img_idx = 10
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"

def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [6]:
state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config_cs = yaml.safe_load(f)

target_img_size = (640, 640)
num_classes_final = 19

data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2,
    check_empty_targets=True,
    img_size = target_img_size,
    **data_module_kwargs
)
data.setup()

In [7]:
net_cfg = config_cs["model"]["init_args"]["network"]
encoder_cfg = net_cfg["init_args"]["encoder"]

#Initialize Encoder
enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
encoder = getattr(importlib.import_module(enc_mod), enc_cls)(img_size=(640,640), **encoder_cfg.get("init_args", {}))

# Initialize EoMT
net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
network = getattr(importlib.import_module(net_mod), net_cls)(
    masked_attn_enabled=True,
    num_classes=19,
    encoder=encoder,
    num_q=200,
    num_blocks=3
)

# Setup Lightning Module
lit_mod, lit_cls = config_cs["model"]["class_path"].rsplit(".", 1)
model_kwargs_final = {k: v for k, v in config_cs["model"]["init_args"].items() if k != "network"}
model_kwargs_final.pop("num_classes", None)
model_kwargs_final.pop("img_size", None)

model = getattr(importlib.import_module(lit_mod), lit_cls)(
          img_size=target_img_size,
          num_classes=num_classes_final,
          network=network,
          **model_kwargs_final,
          ckpt_path=state_dict_path,
          load_ckpt_class_head=False
      ).to(device)


for param in model.parameters():  #freezing
    param.requires_grad = False

# unfreeze head + last Decoder block
layers_to_train = ["class_head",
                   "mask_head",
                   "network.upscale",
                   #"network.q"
                   ]
for name, param in model.named_parameters():
    if any(k in name for k in layers_to_train):
        param.requires_grad = True
        print(f"SBLOCCATO: {name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


SBLOCCATO: network.class_head.weight
SBLOCCATO: network.class_head.bias
SBLOCCATO: network.mask_head.0.weight
SBLOCCATO: network.mask_head.0.bias
SBLOCCATO: network.mask_head.2.weight
SBLOCCATO: network.mask_head.2.bias
SBLOCCATO: network.mask_head.4.weight
SBLOCCATO: network.mask_head.4.bias
SBLOCCATO: network.upscale.0.conv1.weight
SBLOCCATO: network.upscale.0.conv1.bias
SBLOCCATO: network.upscale.0.conv2.weight
SBLOCCATO: network.upscale.0.norm.weight
SBLOCCATO: network.upscale.0.norm.bias
SBLOCCATO: network.upscale.1.conv1.weight
SBLOCCATO: network.upscale.1.conv1.bias
SBLOCCATO: network.upscale.1.conv2.weight
SBLOCCATO: network.upscale.1.norm.weight
SBLOCCATO: network.upscale.1.norm.bias


In [8]:
print(model)

MaskClassificationSemantic(
  (network): EoMT(
    (encoder): ViT(
      (backbone): VisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          (norm): Identity()
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (patch_drop): Identity()
        (norm_pre): Identity()
        (blocks): Sequential(
          (0): Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): Attention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise

In [9]:
for name, param in model.named_parameters():
    print(f"{name:<60} | Richiede Gradiente: {param.requires_grad}")

network.encoder.backbone.cls_token                           | Richiede Gradiente: False
network.encoder.backbone.reg_token                           | Richiede Gradiente: False
network.encoder.backbone.pos_embed                           | Richiede Gradiente: False
network.encoder.backbone.patch_embed.proj.weight             | Richiede Gradiente: False
network.encoder.backbone.patch_embed.proj.bias               | Richiede Gradiente: False
network.encoder.backbone.blocks.0.norm1.weight               | Richiede Gradiente: False
network.encoder.backbone.blocks.0.norm1.bias                 | Richiede Gradiente: False
network.encoder.backbone.blocks.0.attn.qkv.weight            | Richiede Gradiente: False
network.encoder.backbone.blocks.0.attn.qkv.bias              | Richiede Gradiente: False
network.encoder.backbone.blocks.0.attn.proj.weight           | Richiede Gradiente: False
network.encoder.backbone.blocks.0.attn.proj.bias             | Richiede Gradiente: False
network.encoder.backb

# Transfer Learning and Fine Tuning

In [10]:
from lightning.pytorch.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints",
    filename="eomt-cityscapes-finetuning-head_complete-{epoch:02d}-{metrics/val_iou_all:.3f}",
    monitor="metrics/val_iou_all",
    mode="max",
    save_top_k=1,
    save_last = True,
    verbose=True
)

current_max_epochs = 10
use_checkpoint = True
resume_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last-v3.ckpt"


# Configure trainer
trainer = L.Trainer(
    max_epochs=current_max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[checkpoint_callback],


    log_every_n_steps=10,
    default_root_dir="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints"
)

# Starting training
if use_checkpoint:
  print(f"Load training from checkpoint: {resume_ckpt_path}")
  trainer.fit(model, datamodule=data, ckpt_path=resume_ckpt_path)
else:
  trainer.fit(model, datamodule=data)


print(f"Best model saved in: {checkpoint_callback.best_model_path}")
print(f"Best obtained mIoU: {checkpoint_callback.best_model_score:.4f}")



INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


Load training from checkpoint: /content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last-v3.ckpt


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1anujPehiSRqow-CwUA580wxEa24htbA1/CourseProjectAnomaly/checkpoints exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last-v3.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last-v3.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
INFO: 
  | Name      | Type                   | Params | Mode 
-------------------------------------------------------------
0 | network   | EoMT                   

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Best model saved in: /content/drive/.shortcut-targets-by-id/1anujPehiSRqow-CwUA580wxEa24htbA1/CourseProjectAnomaly/checkpoints/eomt-cityscapes-finetuning-head_complete-epoch=03-metrics/val_iou_all=0.703.ckpt
Best obtained mIoU: 0.7029


In [11]:
def infer_semantic(img, target, model, target_size=(640, 640)):
    model.eval()

    if len(img.shape) == 4 and img.shape[0] == 1:  # removing initial batch to give the model the correct dimensions
        img = img.squeeze(0)

    img = img.to(device)
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img]
        img_sizes = [img.shape[-2:]]

        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_layers, class_logits_layers = model(crops) # Forward pass

        m_logits = F.interpolate(mask_logits_layers[-1], target_size, mode="bilinear")
        c_logits = class_logits_layers[-1]

        # query-mask -> pixel-logits
        crop_pixel_logits = model.to_per_pixel_logits_semantic(m_logits, c_logits)
        full_logits = model.revert_window_logits_semantic(crop_pixel_logits, origins, img_sizes)

        preds = full_logits[0].argmax(0).cpu().numpy()

    # Ground Truth conversion to compare
    target_pixel = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    return preds.squeeze().astype(np.int32), target_pixel.squeeze().astype(np.int32)




def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    img_np = img.permute(1, 2, 0).cpu().numpy()

    if img_np.max() > 1.0 or img_np.min() < 0:  #normalising values for visualization
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    axes[0].imshow(img_np)
    axes[0].set_title("Original Image")

    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Model Prediction")

    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Ground Truth")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# Evaluation

### Evaluation for the fine-tuned model (only class_head unlocked)





In [12]:
import numpy as np
import torch
import gc
from tqdm import tqdm

def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))

num_eval_classes = 19
val_loader = data.val_dataloader()

hist_finetuned = np.zeros((num_eval_classes, num_eval_classes))

# Evaluating fine-tuned model
print(f"Evaluation on the fine-tuned model on {len(val_loader.dataset)} images")

model = model.to(device)
model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Fine-tuned Evaluation"):
        imgs, targets = batch

        for j in range(len(imgs)):
            pred_ft, target_ft = infer_semantic(
                imgs[j],
                targets[j],
                model,
                target_size=target_img_size
            )
            hist_finetuned += fast_hist(target_ft.flatten(), pred_ft.flatten(), num_eval_classes)

# Computing final metrics (IoU e mIoU)
ious_finetuned = per_class_iu(hist_finetuned)
miou_finetuned = np.nanmean(ious_finetuned)

classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"{'CLASS CITYSCAPES':<22} | {'IoU FINE-TUNED %':<15}")
print("-" * 30)

for name, iou_ft in zip(classes, ious_finetuned):
    ft_p = iou_ft * 100
    print(f"{name:<22} | {ft_p:>14.2f}%")

print("-" * 30)
print(f"{'MEAN IoU GLOBAL':<22} | {miou_finetuned*100:>14.2f}%")
print("="*30)

Evaluation on the fine-tuned model on 500 images


Fine-tuned Evaluation: 100%|██████████| 250/250 [03:08<00:00,  1.32it/s]


CLASS CITYSCAPES       | IoU FINE-TUNED %
------------------------------
road                   |          96.41%
sidewalk               |          73.48%
building               |          87.98%
wall                   |          54.18%
fence                  |          49.07%
pole                   |          14.66%
traffic light          |          56.95%
traffic sign           |          38.05%
vegetation             |          87.76%
terrain                |          55.62%
sky                    |          90.01%
person                 |          65.34%
rider                  |          18.52%
car                    |          89.24%
truck                  |          63.45%
bus                    |          81.52%
train                  |          66.41%
motorcycle             |          58.21%
bicycle                |          68.44%
------------------------------
MEAN IoU GLOBAL        |          63.96%


### Evaluation for the fine-tuned model (layers unlocked -> class_head + mask_head + upscale)



In [14]:
# head completely unfreezed

import numpy as np
import torch
import gc
from tqdm import tqdm

def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))

num_eval_classes = 19
val_loader = data.val_dataloader()

hist_finetuned = np.zeros((num_eval_classes, num_eval_classes))

# evaluating fine-tuned model
print(f"Evaluation on the fine-tuned model on {len(val_loader.dataset)} images")

model = model.to(device)
model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Fine-tuned Evaluation"):
        imgs, targets = batch

        for j in range(len(imgs)):
            pred_ft, target_ft = infer_semantic(
                imgs[j],
                targets[j],
                model,
                target_size=target_img_size
            )
            hist_finetuned += fast_hist(target_ft.flatten(), pred_ft.flatten(), num_eval_classes)

# Computing final metrics (IoU e mIoU)
ious_finetuned = per_class_iu(hist_finetuned)
miou_finetuned = np.nanmean(ious_finetuned)

classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"{'CLASS CITYSCAPES':<22} | {'IoU FINE-TUNED %':<15}")
print("-" * 30)

for name, iou_ft in zip(classes, ious_finetuned):
    ft_p = iou_ft * 100
    print(f"{name:<22} | {ft_p:>14.2f}%")

print("-" * 30)
print(f"{'MEAN IoU GLOBAL':<22} | {miou_finetuned*100:>14.2f}%")
print("="*30)

Evaluation on the fine-tuned model on 500 images


Fine-tuned Evaluation: 100%|██████████| 250/250 [02:59<00:00,  1.39it/s]


CLASS CITYSCAPES       | IoU FINE-TUNED %
------------------------------
road                   |          97.67%
sidewalk               |          81.58%
building               |          91.92%
wall                   |          59.07%
fence                  |          54.81%
pole                   |          52.54%
traffic light          |          63.90%
traffic sign           |          71.84%
vegetation             |          91.36%
terrain                |          64.88%
sky                    |          94.44%
person                 |          75.17%
rider                  |          27.84%
car                    |          92.62%
truck                  |          72.18%
bus                    |          68.28%
train                  |          39.29%
motorcycle             |          54.04%
bicycle                |          74.62%
------------------------------
MEAN IoU GLOBAL        |          69.90%
